# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

In [0]:
df.display()

# Silver Transformations

## Triming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
df.display()

## Normalization

In [0]:
df = (
    df
    .withColumn("cst_marital_status", 
        F.when(F.upper(col("cst_marital_status")) == "M", "Married")
        .when(F.upper(col("cst_marital_status")) == "S", "Single")
        .otherwise("n/a"))
    .withColumn("cst_gndr", 
        F.when(F.upper(col("cst_gndr")) == "M", "Male")
        .when(F.upper(col("cst_gndr")) == "F", "Female")
        .otherwise('n/a'))
)


In [0]:
df.display()

## Remove Records with Missing Customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

## Renaming columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of dataframe

In [0]:
df.limit(10).display()

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

# Sanity checks of silver table

In [0]:
%sql
SELECT *
FROM workspace.silver.crm_customers
LIMIT 10